# NB-A3 - FaceForensics++ full extraction (real + 4 manipulation families)

Replaces `A2-ffpp-extract.ipynb`, which cached **only** FF++ real videos for the
cut SBI stage.

**Why this exists.** CrossFuse v5 scores 0.95 in-domain but 0.58 (DFDC) / 0.64
(Celeb-DF v2) zero-shot. FakeAVCeleb is ~97% Wav2Lip - a *mouth-region*
reenactment - while Celeb-DF and DFDC are *full-face identity swaps*. The v5
video head never saw a face swap during training, so the gap is a training-data
problem before it is a modelling one. FF++ supplies four swap / reenactment
families (Deepfakes, Face2Face, FaceSwap, NeuralTextures).

**Output** -> publish `/kaggle/working/ffpp_crops_v5` as Kaggle Dataset
`ffpp-crops-v5`, then attach it to NB-B1 and NB-B.

**Attach:** `crossfuse-v5-lib`, plus a FaceForensics++ c23 dataset containing
BOTH `original_sequences/` and `manipulated_sequences/`.

**Status (2026-09): ran.** 5,000 videos (1,000 each of original, Deepfakes, Face2Face, FaceSwap, NeuralTextures), 16 face-crop frames per video at 256 px, published as `ffpp-crops-v5`. `discover_ffpp_videos` handles both the canonical layout and flat mirrors such as `xdxd003/ff-c23` (`FaceForensics++_C23/{original,Deepfakes,...}`).

In [ ]:
!pip install -q --no-deps facenet-pytorch
import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))
print("Using crossfuse_v5 library from:", _lib_hits[0])

In [ ]:
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from crossfuse_v5 import (
    CONFIG, build_detector, extract_face_crops_contiguous, make_rng,
)
from ffpp_v5 import (
    FFPP_METHODS, FFPP_CONFIG, discover_ffpp_videos, build_ffpp_manifest,
)

N_DEVICES = max(1, torch.cuda.device_count())
DEVICE_IDS = [f"cuda:{i}" for i in range(N_DEVICES)] if torch.cuda.is_available() else ["cpu"]
print(f"Extraction devices: {DEVICE_IDS}")

OUT_DIR = "/kaggle/working/ffpp_crops_v5"
os.makedirs(OUT_DIR, exist_ok=True)
MANIFEST = os.path.join(OUT_DIR, "manifest.csv")

# The crop convention is READ from CONFIG, not redeclared. extract_face_crops_contiguous
# reads CONFIG["FACE_MARGIN"] internally, so a local override here would be dead code
# that merely LOOKS like it changed the crop -- which is exactly what A2 did with its
# unused FACE_MARGIN = 0.45. These crops must match the FakeAVCeleb cache exactly or
# the pretrained encoder will not transfer.
print(f"crop {CONFIG['CROP_SIZE']}px | margin {CONFIG['FACE_MARGIN']} | "
      f"{FFPP_CONFIG['FRAMES_PER_VIDEO']} frames/video")

In [ ]:
# ---- Diagnostic (optional): print the folder tree of everything attached ----
# Use this if discover_ffpp_videos below reports 0 videos for a family. It handles both the
# canonical layout (original_sequences/.../c23/videos) and flat mirrors (e.g. xdxd003/ff-c23's
# FaceForensics++_C23/{original,Deepfakes,...}); this walk shows which layout you actually
# have. MAX_DEPTH must be >= 9 to see below Kaggle's input/datasets/<owner>/<slug>/ prefix.
import os

ROOT = "/kaggle/input"
MAX_DEPTH = 9
for r, dirs, files in os.walk(ROOT):
    depth = r[len(ROOT):].count(os.sep)
    if depth >= MAX_DEPTH:
        dirs[:] = []
        continue
    mp4s = [f for f in files if f.lower().endswith(".mp4")]
    tag = f"  <- {len(mp4s)} .mp4 files" if mp4s else ""
    print("  " * depth + os.path.basename(r) + tag)

print()
print("If a family shows 0 videos in discover_ffpp_videos below, compare the folder names above "
      "with the patterns in ffpp_v5.discover_ffpp_videos (original_sequences/, manipulated_sequences/, "
      "or a flat original/ + <Method>/ layout) and attach a mirror that has all five families.")

In [ ]:
# ---- 1. Discover, and FAIL LOUDLY if a manipulation family is missing ----
found = discover_ffpp_videos(roots=("/kaggle/input",), compression="c23", verbose=True)

missing = [k for k in ("original",) + FFPP_METHODS if not found[k]]
assert not missing, (
    f"FF++ families not found: {missing}. Training on 3 of 4 forgery types "
    f"silently weakens generalisation, so this is a hard stop rather than a "
    f"warning. Check the attached dataset exposes "
    f"manipulated_sequences/<Method>/c23/videos/*.mp4 -- the xdxd003/ff-c23 "
    f"Kaggle dataset has the full tree.")

rows = build_ffpp_manifest(found, seed=CONFIG["SEED"], verbose=True)
est_gb = len(rows) * FFPP_CONFIG["FRAMES_PER_VIDEO"] * CONFIG["CROP_SIZE"] ** 2 * 3 / 1e9
print(f"\nestimated crop storage: {est_gb:.2f} GB (Kaggle output limit is 20 GB)")
assert est_gb < 19.0, "would overflow Kaggle's output cap -- lower FRAMES_PER_VIDEO"

In [ ]:
# ---- 2. Extraction: one detector per GPU, jobs round-robined via a thread pool ----
def process_job(job, mtcnn_local):
    out_path = os.path.join(OUT_DIR, job["key"] + ".npy")
    if os.path.exists(out_path):
        return job, "cached"
    rng = make_rng(job["key"])
    try:
        crops, valid, fps, start, ts = extract_face_crops_contiguous(
            job["path"], mtcnn_local, rng=rng,
            n_frames=FFPP_CONFIG["DETECT_FRAMES"], out_size=CONFIG["CROP_SIZE"])
    except Exception as e:
        return job, f"error:{type(e).__name__}"
    if crops is None or int(valid.sum()) < FFPP_CONFIG["FRAMES_PER_VIDEO"]:
        return job, "too_few_faces"
    # Keep ONLY frames where a face was actually detected. On failure
    # extract_face_crops_contiguous falls back to a centre crop, and for a
    # manipulated clip that crop may not contain the manipulated face at all --
    # a mislabelled training sample, not merely a noisy one.
    idx = np.where(valid)[0]
    pick = sorted(rng.sample(list(idx), FFPP_CONFIG["FRAMES_PER_VIDEO"]))
    np.save(out_path, crops[pick])
    return job, "ok"


detectors = {d: build_detector(d) for d in DEVICE_IDS}
# A thread pool does not pin a job to a worker. Protect each shared detector.
import threading
detector_locks = {d: threading.Lock() for d in DEVICE_IDS}
_process_job_unlocked = process_job
def process_job(job, *args):
    detector = args[-1]
    device_id = next(d for d, instance in detectors.items() if instance is detector)
    with detector_locks[device_id]:
        return _process_job_unlocked(job, *args)
status_counts, kept = {}, []
with ThreadPoolExecutor(max_workers=len(DEVICE_IDS)) as pool:
    futs = [pool.submit(process_job, job, detectors[DEVICE_IDS[i % len(DEVICE_IDS)]])
            for i, job in enumerate(rows)]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="FF++ crops"):
        job, status = fut.result()
        status_counts[status] = status_counts.get(status, 0) + 1
        if status in ("ok", "cached"):
            kept.append(job)

print(f"\nStatus: {status_counts}")
print(f"Kept {len(kept)}/{len(rows)} videos")

In [ ]:
# ---- 3. Manifest write + integrity check ----
per_method = {}
for r in kept:
    per_method[r["method"]] = per_method.get(r["method"], 0) + 1
print("Per-method survivors:", per_method)
for m in FFPP_METHODS:
    assert per_method.get(m, 0) > 0, (
        f"every {m} clip failed extraction -- investigate before training, a "
        f"missing family is invisible once the manifest is written")

kept.sort(key=lambda r: r["key"])
fields = ["key", "path", "method", "label", "target_id", "source_id", "identity_group"]
with open(MANIFEST, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows([{k: r[k] for k in fields} for r in kept])

on_disk = {os.path.splitext(os.path.basename(p))[0]
           for p in glob.glob(os.path.join(OUT_DIR, "*.npy"))}
keys = {r["key"] for r in kept}
assert len(keys) == len(kept), "duplicate keys in manifest"
assert not (keys - on_disk), f"{len(keys - on_disk)} manifest rows have no .npy on disk"

total = sum(os.path.getsize(p) for p in glob.glob(os.path.join(OUT_DIR, "*.npy")))
n_fake = sum(r["label"] for r in kept)
print(f"\nManifest OK -> {len(kept)} rows | real/fake {len(kept) - n_fake}/{n_fake} "
      f"| {total / 1e9:.2f} GB")

fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
by_m = {}
for r in kept:
    by_m.setdefault(r["method"], r)
for ax, m in zip(axes, ("original",) + FFPP_METHODS):
    if m in by_m:
        arr = np.load(os.path.join(OUT_DIR, by_m[m]["key"] + ".npy"), mmap_mode="r")
        ax.imshow(arr[0])
        ax.set_title(f"{m}\nlabel {by_m[m]['label']}", fontsize=9)
    ax.axis("off")
plt.suptitle("FF++ cached crop spot-check (one per family)", fontsize=11)
plt.tight_layout(); plt.show()

print("\nStage A3 complete. Save Version, publish /kaggle/working/ffpp_crops_v5 "
      "as Kaggle Dataset 'ffpp-crops-v5', then attach it to NB-B1 and NB-B.")